# 文档解析实验: PDF文本/表格/图像提取

> **学习目标**: 理解PDF文档结构，掌握PyMuPDF和pdfplumber库，学会使用智析项目的 `doc_parser.py` 模块
>
> **对应项目模块**: `src/doc_parser.py` (CV层)
>
> **预计时间**: 2-3天
>
> **前置条件**: `pip install PyMuPDF pdfplumber Pillow tqdm`

---

## 背景知识

### PDF文档是什么？
PDF（Portable Document Format）不只是"一张图片"，它是一个**结构化的容器**，内部包含:
- **文本对象**: 带有字体、大小、位置信息的文字
- **图像对象**: 嵌入的图片（JPEG/PNG等）
- **矢量图形**: 线条、表格框线
- **页面结构**: 每页有独立的坐标系

### 为什么要解析PDF？
在智析项目中，PDF解析是**整个系统的入口**:
```
PDF文档 → [文档解析] → 纯文本+表格+图像 → [NLP分析] → [知识图谱] → [RAG问答]
```
如果解析不好，后续所有模块都会受影响。所以这一步非常重要！

## Part 1: PyMuPDF基础 — 直接操作PDF

### 什么是PyMuPDF？
PyMuPDF（导入名为 `fitz`）是Python中最快最全面的PDF处理库。它可以:
- 提取文本（包括中文）
- 提取嵌入图像
- 获取页面尺寸和布局信息
- 创建和修改PDF

In [ ]:
import fitz  # PyMuPDF的导入名是fitz
import os

print(f"PyMuPDF 版本: {fitz.version}")

# ========================================
# 1.1 创建一个测试PDF (方便后续实验)
# ========================================
os.makedirs('../data/sample_docs', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

doc = fitz.open()

# 第1页: 项目介绍
page = doc.new_page()
# insert_text(位置, 文本, 字号)  位置是(x, y)坐标，单位是"点"(1点=1/72英寸)
page.insert_text((72, 72), '智析 (ZhiXi) - 多模态文档智能分析平台', fontsize=20)
page.insert_text((72, 110), '项目概述:', fontsize=14)
page.insert_text((72, 135), '智析是一个能够自动解析、理解、检索和问答行业研究报告的智能平台。', fontsize=11)
page.insert_text((72, 155), '用户上传PDF研报，系统自动提取信息、建立知识索引，', fontsize=11)
page.insert_text((72, 175), '用户可以通过自然语言对话获取洞察。', fontsize=11)
page.insert_text((72, 210), '技术栈:', fontsize=14)
page.insert_text((72, 235), '1. 文档解析层 (CV): PyMuPDF + pdfplumber + PaddleOCR', fontsize=11)
page.insert_text((72, 255), '2. NLP分析层: HuggingFace Transformers + spaCy + KeyBERT', fontsize=11)
page.insert_text((72, 275), '3. 知识构建层 (数据挖掘): NetworkX + Scikit-learn', fontsize=11)
page.insert_text((72, 295), '4. RAG问答层 (LLM): LangChain + ChromaDB + OpenAI/Ollama', fontsize=11)
page.insert_text((72, 315), '5. Web界面: Streamlit', fontsize=11)

# 第2页: 系统架构
page2 = doc.new_page()
page2.insert_text((72, 72), '系统架构设计', fontsize=20)
page2.insert_text((72, 110), '采用四层架构，每层职责明确:', fontsize=12)
page2.insert_text((72, 140), '文档解析层 → NLP分析层 → 知识构建层 → RAG问答层', fontsize=14)
page2.insert_text((72, 180), '核心设计原则:', fontsize=14)
page2.insert_text((72, 210), '- 延迟加载: 模型按需初始化，节省内存', fontsize=11)
page2.insert_text((72, 230), '- 降级策略: 每个模块都有错误处理，保证系统不崩溃', fontsize=11)
page2.insert_text((72, 250), '- 模块化: 每个层可以独立开发和测试', fontsize=11)
page2.insert_text((72, 290), '关键数据指标:', fontsize=14)
page2.insert_text((72, 315), '- 支持PDF页数: 1-1000页', fontsize=11)
page2.insert_text((72, 335), '- 文本提取准确率: >95%', fontsize=11)
page2.insert_text((72, 355), '- 问答响应时间: <3秒', fontsize=11)

# 第3页: 更多细节
page3 = doc.new_page()
page3.insert_text((72, 72), 'RAG检索增强生成', fontsize=20)
page3.insert_text((72, 110), 'RAG (Retrieval-Augmented Generation) 是当前最流行的LLM应用架构。', fontsize=12)
page3.insert_text((72, 140), '核心流程:', fontsize=14)
page3.insert_text((72, 170), '1. 文档切块 (Chunking): 将长文档分成500字符的小块', fontsize=11)
page3.insert_text((72, 190), '2. 向量化 (Embedding): 将文本块转换为高维向量', fontsize=11)
page3.insert_text((72, 210), '3. 存储 (Indexing): 将向量存入ChromaDB向量数据库', fontsize=11)
page3.insert_text((72, 230), '4. 检索 (Retrieval): 用户提问时，找到最相关的文档块', fontsize=11)
page3.insert_text((72, 250), '5. 生成 (Generation): 将检索到的内容作为上下文，让LLM生成回答', fontsize=11)

test_pdf_path = '../data/sample_docs/test.pdf'
doc.save(test_pdf_path)
doc.close()
print(f"测试PDF已创建: {test_pdf_path}")
print(f"文件大小: {os.path.getsize(test_pdf_path)} bytes")

In [ ]:
# ========================================
# 1.2 打开PDF并提取文本 (最基础的操作)
# ========================================

doc = fitz.open(test_pdf_path)

print(f"文件: {doc.name}")
print(f"页数: {len(doc)}")
print(f"元数据: {doc.metadata}")
print()

# 逐页提取文本
for page_num, page in enumerate(doc):
    text = page.get_text("text")  # "text"模式提取纯文本
    print(f"{'='*50}")
    print(f"第 {page_num + 1} 页 (尺寸: {page.rect.width:.0f} x {page.rect.height:.0f} 点)")
    print(f"{'='*50}")
    print(text[:300])  # 只看前300字符
    print()

doc.close()

In [ ]:
# ========================================
# 1.3 不同的文本提取模式
# ========================================

doc = fitz.open(test_pdf_path)
page = doc[0]  # 第1页

# 模式1: "text" — 纯文本 (最常用)
text_mode = page.get_text("text")
print("=== 模式: text ===")
print(text_mode[:200])

# 模式2: "dict" — 结构化信息 (包含字体、颜色、位置)
dict_mode = page.get_text("dict")
print(f"\n=== 模式: dict (包含 {len(dict_mode['blocks'])} 个文本块) ===")
for block in dict_mode['blocks'][:2]:
    if 'lines' in block:
        for line in block['lines'][:1]:
            for span in line['spans'][:1]:
                print(f"  文本: '{span['text'][:50]}'")
                print(f"  字体: {span['font']}, 大小: {span['size']:.1f}")
                print(f"  位置: ({span['bbox'][0]:.0f}, {span['bbox'][1]:.0f})")

# 模式3: "blocks" — 按文本块提取 (保留布局)
blocks = page.get_text("blocks")
print(f"\n=== 模式: blocks ({len(blocks)} 个块) ===")
for block in blocks[:3]:
    print(f"  位置: ({block[0]:.0f},{block[1]:.0f})-({block[2]:.0f},{block[3]:.0f}) | 文本: {block[4][:60].strip()}")

doc.close()

In [ ]:
# ========================================
# 1.4 提取PDF中的嵌入图像
# ========================================

# 先创建一个带图片的PDF
doc_with_img = fitz.open()
page = doc_with_img.new_page()
page.insert_text((72, 72), '这是一份带图片的文档', fontsize=16)

# 创建一个简单的彩色图像并嵌入
from PIL import Image
import io

img = Image.new('RGB', (200, 100), color='#4ECDC4')
img_bytes = io.BytesIO()
img.save(img_bytes, format='PNG')
img_bytes.seek(0)

page.insert_image(fitz.Rect(72, 120, 272, 220), stream=img_bytes.getvalue())

img_pdf_path = '../data/sample_docs/with_image.pdf'
os.makedirs(os.path.dirname(img_pdf_path), exist_ok=True)
doc_with_img.save(img_pdf_path)
doc_with_img.close()
print(f"带图片PDF已创建: {img_pdf_path}")

# 提取图像
doc = fitz.open(img_pdf_path)
page = doc[0]
image_list = page.get_images(full=True)
print(f"\n发现 {len(image_list)} 张图像")

os.makedirs('../data/processed/test_images', exist_ok=True)
for img_idx, img_info in enumerate(image_list):
    xref = img_info[0]
    base_image = doc.extract_image(xref)
    img_bytes = base_image["image"]
    img_ext = base_image.get("ext", "png")
    
    save_path = f'../data/processed/test_images/img_{img_idx + 1}.{img_ext}'
    with open(save_path, 'wb') as f:
        f.write(img_bytes)
    
    print(f"  图像{img_idx+1}: {len(img_bytes)} bytes, 格式: {img_ext}")
    print(f"  保存至: {save_path}")

doc.close()

## Part 2: pdfplumber — 表格提取

### 为什么需要pdfplumber？
PyMuPDF擅长提取文本和图像，但对**表格**的处理不够好。pdfplumber专门用于PDF表格提取，它能:
- 识别表格的行列结构
- 处理合并单元格
- 提取为结构化的Python列表

In [ ]:
import pdfplumber

# 创建一个带表格的PDF
doc = fitz.open()
page = doc.new_page()
page.insert_text((72, 50), 'AI行业研究报告 - 数据表格', fontsize=16)

# 用PyMuPDF画一个简单的表格
# (实际项目中PDF已经有表格了，这里是为了演示)
table_data = [
    ['指标', '2023年', '2024年', '增长率'],
    ['全球AI市场规模', '$1500亿', '$2000亿', '33%'],
    ['LLM应用数量', '5000+', '20000+', '300%'],
    ['AI工程师需求', '100K', '250K', '150%'],
    ['开源模型数量', '500+', '2000+', '300%'],
]

y_start = 90
row_height = 25
col_widths = [150, 80, 80, 80]
x_start = 72

for row_idx, row in enumerate(table_data):
    x = x_start
    y = y_start + row_idx * row_height
    for col_idx, cell in enumerate(row):
        # 画单元格边框
        rect = fitz.Rect(x, y, x + col_widths[col_idx], y + row_height)
        page.draw_rect(rect, color=(0, 0, 0), width=0.5)
        # 写入文本
        fontsize = 10 if row_idx == 0 else 9
        page.insert_text((x + 5, y + 17), cell, fontsize=fontsize)
        x += col_widths[col_idx]

table_pdf_path = '../data/sample_docs/with_table.pdf'
doc.save(table_pdf_path)
doc.close()
print(f"带表格PDF已创建")

# 用pdfplumber提取表格
with pdfplumber.open(table_pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    print(f"\n发现 {len(tables)} 个表格")
    
    for t_idx, table in enumerate(tables):
        print(f"\n=== 表格 {t_idx + 1} ({len(table)} 行) ===")
        for row in table:
            print(f"  {row}")
        
        # 转换为Pandas DataFrame
        if table:
            import pandas as pd
            headers = table[0]
            data = table[1:]
            df_table = pd.DataFrame(data, columns=headers)
            print(f"\n--- DataFrame格式 ---")
            print(df_table)

## Part 3: 使用智析项目的 doc_parser.py

现在你已经理解了PDF解析的底层原理，让我们来使用项目中封装好的模块。

### `doc_parser.py` 的设计思路:
1. **DocumentParser类**: 封装了PyMuPDF + pdfplumber的调用
2. **parse()方法**: 一次调用完成所有提取（文本+表格+图像）
3. **get_text_chunks()方法**: 将文本切分为适合RAG的块
4. **DocumentResult**: 统一的结果数据结构

In [ ]:
import sys
sys.path.insert(0, '..')

from src.doc_parser import DocumentParser, parse_pdf

# ========================================
# 3.1 一键解析PDF
# ========================================

# 使用封装好的函数
result = parse_pdf(test_pdf_path, output_dir='../data/processed', save_result=True)

print(f"文件名: {result.filename}")
print(f"页数: {result.total_pages}")
print(f"总字符数: {len(result.full_text)}")
print(f"\n每页内容预览:")
for page in result.pages:
    print(f"  第{page.page_number}页: {len(page.text)}字符, {len(page.tables)}个表格, {len(page.images)}张图像")

In [ ]:
# ========================================
# 3.2 文本切块 — RAG的关键步骤
# ========================================
# 
# 为什么需要切块？
# 1. LLM有上下文长度限制 (GPT-4o: 128K tokens)
# 2. 嵌入向量对短文本效果更好
# 3. 检索时需要精确匹配到相关段落
#
# chunk_size: 每个块的字符数 (太小会丢失上下文，太大会降低检索精度)
# chunk_overlap: 相邻块的重叠字符 (防止关键信息被切断)

parser = DocumentParser(test_pdf_path, output_dir='../data/processed')

# 尝试不同的切块策略
for chunk_size in [200, 500, 1000]:
    chunks = parser.get_text_chunks(chunk_size=chunk_size, chunk_overlap=50)
    print(f"\n=== chunk_size={chunk_size} ===")
    print(f"切分为 {len(chunks)} 个块")
    for i, chunk in enumerate(chunks[:3]):  # 只看前3个
        print(f"  块{i} (页{chunk['page']}): {chunk['text'][:80]}...")
    print(f"  ...")

In [ ]:
# ========================================
# 3.3 完整的端到端流程演示
# ========================================

print("=" * 60)
print("智析 - 文档解析完整流程演示")
print("=" * 60)

# Step 1: 解析
print("\n[Step 1] 解析PDF...")
result = parse_pdf(test_pdf_path, save_result=False)

# Step 2: 查看统计
print(f"\n[Step 2] 解析统计:")
print(f"  总页数: {result.total_pages}")
print(f"  总字符: {len(result.full_text)}")
tables = sum(len(p.tables) for p in result.pages)
images = sum(len(p.images) for p in result.pages)
print(f"  表格数: {tables}")
print(f"  图像数: {images}")

# Step 3: 切块
print(f"\n[Step 3] 文本切块...")
chunks = parser.get_text_chunks(chunk_size=500, chunk_overlap=50)
print(f"  切分结果: {len(chunks)} 个块")
avg_len = sum(len(c['text']) for c in chunks) / len(chunks)
print(f"  平均块长度: {avg_len:.0f} 字符")

# Step 4: 保存结果
print(f"\n[Step 4] 保存解析结果...")
result.save('../data/processed/parse_result.json')

print(f"\n{'=' * 60}")
print("文档解析完成! 下一步: 运行 03_rag_experiment.ipynb 体验RAG问答")

---
## 知识总结

### 你学到了什么:
1. **PDF结构**: 文本对象、图像对象、页面坐标系
2. **PyMuPDF (fitz)**: 文本提取（3种模式）、图像提取、页面信息
3. **pdfplumber**: 表格识别和结构化提取
4. **doc_parser.py**: 项目封装的解析模块，一键完成全流程
5. **文本切块**: chunk_size和chunk_overlap的含义和调优

### 核心概念:
| 概念 | 含义 | 在智析中的作用 |
|------|------|--------------|
| `page.get_text("text")` | 提取纯文本 | 获取文档内容 |
| `page.get_images()` | 获取图像列表 | 提取图表用于分析 |
| `page.extract_tables()` | 提取表格结构 | 获取数据表格 |
| `chunk_size` | 文本块大小 | 影响RAG检索精度 |
| `chunk_overlap` | 块间重叠 | 防止信息丢失 |

### 下一步:
- 用**真实论文/研报PDF**测试解析效果
- 运行 `04_nlp_analysis.ipynb` 学习NLP分析
- 运行 `03_rag_experiment.ipynb` 体验智能问答